# Maze LMDP workflows

This notebook is the supported end-to-end walkthrough: arbitrary maze geometry, a flat first-exit solve, a fixed-subgoal hierarchy with online Z-iteration, the passive subgoal graph, and NMF-discovered core-gated soft subgoals with an interactive rollout.

In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from andrew_mlmdp import (
    Environment,
    Maze,
    NMFConfig,
    NMFConnectivityConfig,
    Parameters,
    SubgoalBasis,
    desirability_grid,
    discover_subgoals,
    point_parameters,
    soft_parameters,
)
from andrew_mlmdp import (
    plotting as viz,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

## Maze and shared physical dynamics

In [ ]:
flat_parameters = Parameters()
hard_parameters = point_parameters(upper_control_cost=6.5)
maze = Maze.from_file(PROJECT_ROOT / "mazes" / "four_rooms.txt")
environment = Environment(maze)
goal = (1,9)

print(f"flat parameters: {flat_parameters}")
print(f"hard hierarchy defaults: {hard_parameters}")
print(f"maze shape: {maze.shape}")
print(f"free states: {len(maze.free_cells)}")
print(f"goal state: {maze.state_index(goal)}")

## Exact flat first-exit LMDP

The environment caches the physical passive dynamics. A flat solution contains the desirability, controlled policy, and rollout method for one goal.

In [ ]:
flat = environment.solve(goal, parameters=flat_parameters)
flat_grid = desirability_grid(maze, flat.desirability)
positive = flat_grid[np.isfinite(flat_grid) & (flat_grid > 0.0)]
log_grid = np.where(flat_grid > 0.0, np.log10(flat_grid), np.nan)

figure = go.Figure(go.Heatmap(
    z=log_grid,
    colorscale="Viridis",
    colorbar={"title": "log10 desirability"},
    customdata=flat_grid,
    hovertemplate="desirability: %{customdata:.4g}<extra></extra>",
))
figure.add_trace(go.Scatter(
    x=[goal[1]], y=[goal[0]], mode="markers",
    marker={"symbol": "star", "color": "red", "size": 16},
    name="goal",
))
figure.update_layout(
    title="Exact flat desirability", xaxis_title="column", yaxis_title="row",
    width=700, height=600, template="plotly_white",
)
figure.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)
figure.show()

In [ ]:
viz.plot_controlled_dynamics(maze, flat.controlled, goal=goal).show()

flat_start = (3, 0)
flat_rollout = flat.rollout(flat_start, seed=0)
print(f"reached goal: {flat_rollout[-1] == goal}")
print(f"physical steps: {len(flat_rollout) - 1}")
viz.plot_trajectory(maze, flat_rollout, goal=goal).show()

## Fixed-subgoal two-layer hierarchy

Point subgoals are represented internally as one-hot profile columns. The same hierarchy implementation is used later for distributed profiles.

In [ ]:
subgoal_labels = ("A", "B", "C", "D", "E", "F")
subgoals = (
    (0, 0),
    (9, 2),
    (2, 3),
    (3, 7),
    (9, 7),
    (7, 9),
)
hierarchical_start = (3, 2)
point_basis = SubgoalBasis.from_locations(
    maze, subgoals, labels=subgoal_labels
)
hierarchy = environment.hierarchy(
    point_basis,
    parameters=hard_parameters,
)
task = hierarchy.task(goal)

### Task-independent passive subgoal graph

In [ ]:
subgoal_passive = hierarchy.upper_passive
viz.plot_subgoal_passive_dynamics(
    maze,
    subgoals,
    subgoal_passive,
    labels=subgoal_labels,
).show()
print(np.round(subgoal_passive, 4))

### Goal-conditioned matrices and task composition

In [ ]:
print("target order:", subgoal_labels + ("goal",))
print("lower passive:", task.lower_dynamics.passive.shape)
print("boundary basis Q_b:", task.task_basis.boundary_desirability.shape)
print("interior basis Z_i:", task.task_basis.interior_desirability.shape)
print("upper passive:", task.upper_dynamics.passive.shape)

initial_plan = task.plan(hierarchical_start)
abstract_labels = subgoal_labels + ("goal",)
source_labels = subgoal_labels + ("start",)
passive_matrix = np.vstack(
    (task.upper_dynamics.passive.T, initial_plan.upper_passive)
)
controlled_matrix = np.vstack(
    (task.upper_controlled.T, initial_plan.upper_policy)
)
transition_matrices = (passive_matrix, controlled_matrix)
expected_shape = (len(source_labels), len(abstract_labels))
assert all(matrix.shape == expected_shape for matrix in transition_matrices)
assert all(
    np.allclose(matrix.sum(axis=1), 1.0)
    for matrix in transition_matrices
)
matrix_maximum = max(matrix.max() for matrix in transition_matrices)
figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Layer 2 passive", "Layer 2 controlled"),
)
for column, matrix in enumerate(transition_matrices, 1):
    figure.add_trace(go.Heatmap(
        z=matrix,
        x=abstract_labels,
        y=source_labels,
        zmin=0.0,
        zmax=matrix_maximum,
        coloraxis="coloraxis",
        hovertemplate="%{y} → %{x}: %{z:.4f}<extra></extra>",
    ), row=1, col=column)
    figure.update_xaxes(title_text="next abstract state", row=1, col=column)
    figure.update_yaxes(title_text="current state", row=1, col=column)
figure.update_layout(
    width=1000, height=500, template="plotly_white",
    coloraxis={"colorscale": "Viridis", "colorbar": {"title": "probability"}},
)
figure.show()

In [ ]:
print(" target   passive  controlled  reward    weight")
for label, passive, controlled, reward, weight in zip(
    abstract_labels,
    initial_plan.upper_passive,
    initial_plan.upper_policy,
    initial_plan.rewards,
    initial_plan.weights,
):
    print(
        f" {label:>4}    {passive:7.3f}     {controlled:7.3f}"
        f"    {reward:7.3f}   {weight:7.3f}"
    )

composed_grid = desirability_grid(maze, initial_plan.desirability)
log_composed_grid = np.where(
    composed_grid > 0.0,
    np.log10(composed_grid),
    np.nan,
)
figure = go.Figure(go.Heatmap(
    z=log_composed_grid,
    colorscale="Viridis",
    colorbar={"title": "log10 desirability"},
    customdata=composed_grid,
    hovertemplate="desirability: %{customdata:.4g}<extra></extra>",
))
figure.add_trace(go.Scatter(
    x=[goal[1]], y=[goal[0]], mode="markers",
    marker={"symbol": "star", "color": "red", "size": 16}, name="goal",
))
figure.update_layout(
    title="Composed lower-layer desirability",
    xaxis_title="column", yaxis_title="row",
    width=700, height=600, template="plotly_white",
)
figure.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)
figure.show()

### Exact and online hierarchical rollouts

In [ ]:
hierarchical_rollout = task.rollout(hierarchical_start, seed=0)
print("status:", hierarchical_rollout.status)
print("physical steps:", hierarchical_rollout.physical_steps)
print("zero-time accesses:", hierarchical_rollout.accesses)

figure = viz.plot_trajectory(
    maze, hierarchical_rollout.trajectory, goal=goal
)
figure.add_trace(go.Scatter(
    x=[coordinate[1] for coordinate in subgoals],
    y=[coordinate[0] for coordinate in subgoals],
    mode="markers+text",
    text=subgoal_labels,
    textposition="top right",
    marker={
        "size": 13,
        "color": "rgba(0,0,0,0)",
        "line": {"color": "darkorange", "width": 2},
    },
    textfont={"color": "darkorange"},
    name="subgoals",
))
figure.update_layout(title=(
    f"Hierarchical rollout: {hierarchical_rollout.status} "
    f"({hierarchical_rollout.physical_steps} steps)"
))
figure.show()

In [ ]:
from IPython.display import display

rollout_animation = viz.animate_rollout(
    task,
    hierarchical_start,
    seed=0,
    max_steps=100,
    interval=450,
    subgoal_labels=subgoal_labels,
)
rollout_animation.show()

In [ ]:
online_animation = viz.animate_rollout(
    task,
    hierarchical_start,
    goal_learning="online",
    z_sweeps_per_step=1,
    max_steps=100,
    seed=28,
    interval=450,
    subgoal_labels=subgoal_labels,
)
online_animation.show()

In [ ]:
learned_goal = None
online_episodes = []
for episode_seed in range(5):
    episode = task.rollout(
        hierarchical_start,
        goal_learning="online",
        initial_goal_desirability=learned_goal,
        z_sweeps_per_step=1,
        max_steps=100,
        seed=episode_seed,
    )
    online_episodes.append(episode)
    learned_goal = episode.final_goal_desirability

for index, episode in enumerate(online_episodes, start=1):
    print(
        f"episode {index}: {episode.status}, "
        f"{episode.physical_steps} physical steps, "
        f"{episode.z_iterations} Z sweeps"
    )

### Interactive fixed-subgoal composition

Drag the agent and goal markers to inspect the fixed-basis task blend.

In [ ]:
interactive_figure = viz.explore_subgoal_desirability(
    task,
    hierarchical_start,
    subgoal_labels=subgoal_labels,
)
interactive_figure.show()

## NMF-discovered core-gated soft subgoals

Every requested rank is fitted once. After plotting the diagnostics, choose `soft_rank` below; that cached result and the common rank-independent execution defaults are reused downstream.

In [ ]:
soft_study = discover_subgoals(
    environment,
    ranks=tuple(range(2, 13)),
    parameters=NMFConfig(),
    connectivity=NMFConnectivityConfig(restart_seeds=(1, 2)),
)
viz.plot_rank_diagnostics(soft_study.diagnostics).show()
soft_rank = 4  # change after inspecting the diagnostics
soft_discovery = soft_study.result(soft_rank)
if soft_discovery is None:
    raise RuntimeError("All selected-rank restarts were excluded")
print(
    f"normalized KL error: {soft_discovery.reconstruction_error:.3f}; "
    f"{soft_discovery.n_iter} iterations; "
    f"converged={soft_discovery.converged}"
)
viz.plot_subtasks(soft_discovery).show()

In [ ]:
soft_basis = SubgoalBasis.from_profiles(
    maze,
    soft_discovery.profiles,
    core_threshold=0.8,
    core_exponent=1.0,
)
soft_template = environment.hierarchy(
    soft_basis,
    parameters=soft_parameters(k=soft_rank, upper_control_cost=1.8),
)

### Interactive soft rollout

Drag the green start circle and red goal star to stage new free cells, then press **Recompute rollout**. Recompute builds only the cached goal-conditioned task and samples one trajectory; it does not rerun NMF or reapply the core gate.

In [ ]:
from IPython.display import display

soft_player = viz.explore_rollout(
    soft_template,
    hierarchical_start,
    goal,
    max_steps=100,
    max_abstract_accesses=100,
    seed=0,
)
display(soft_player.controls)
soft_player.figure.show()